# Notebook 2 — Validación Estructural y Semántica
## Caso: Predicción de Default en Solicitudes de Préstamos

### Objetivo
Validar que los datos limpios cumplan reglas técnicas y lógicas
antes de ser cargados a la base de datos.

### Tipos de validación
1. **Estructural** → que los tipos de datos sean correctos
2. **Semántica** → que los valores tengan sentido lógico

### Etapas
1. Cargar dataset limpio
2. Configurar logs
3. Aplicar validaciones estructurales
4. Aplicar validaciones semánticas
5. Separar registros válidos e inválidos
6. Exportar resultados

In [1]:
import pandas as pd
import os
import logging

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


In [2]:
# Crear carpeta logs si no existe
os.makedirs("../logs", exist_ok=True)

# Configurar el sistema de logs para este notebook
logging.basicConfig(
    filename="../logs/validation.log",  # archivo de log separado para validación
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# Registrar inicio del proceso
logging.info("Inicio del proceso de validación")
print("Logger configurado correctamente")

Logger configurado correctamente


In [3]:
# Leer el dataset limpio generado en el Notebook 1
df = pd.read_csv("../data/processed/loans_clean.csv")

# Registrar en el log
logging.info(f"Dataset cargado correctamente: {len(df)} registros")
print(f"Dataset cargado: {len(df)} registros")

# Ver las primeras filas
df.head()

Dataset cargado: 45000 registros


,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22,female,Master,71948,0,RENT,35000,PERSONAL,16.02,0.49,3,561,No,1
1,21,female,High School,12282,0,OWN,1000,EDUCATION,11.14,0.08,2,504,Yes,0
2,25,female,High School,12438,3,MORTGAGE,5500,MEDICAL,12.87,0.44,3,635,No,1
3,23,female,Bachelor,79753,0,RENT,35000,MEDICAL,15.23,0.44,2,675,No,1
4,24,male,Master,66135,1,RENT,35000,MEDICAL,14.27,0.53,4,586,No,1


In [4]:
# Lista de columnas esperadas según el metadata
columnas_esperadas = [
    "person_age", "person_gender", "person_education", "person_income",
    "person_emp_exp", "person_home_ownership", "loan_amnt", "loan_intent",
    "loan_int_rate", "loan_percent_income", "cb_person_cred_hist_length",
    "credit_score", "previous_loan_defaults_on_file", "loan_status"
]

# Verificar que todas las columnas esperadas están presentes
columnas_faltantes = [col for col in columnas_esperadas if col not in df.columns]

if columnas_faltantes:
    logging.warning(f"Columnas faltantes: {columnas_faltantes}")
    print(f"Columnas faltantes: {columnas_faltantes}")
else:
    logging.info("Validación estructural: todas las columnas presentes")
    print("Todas las columnas esperadas están presentes")

# Verificar tipos de datos correctos según metadata
tipos_esperados = {
    "person_age": "int64",
    "person_income": "int64",
    "person_emp_exp": "int64",
    "loan_amnt": "int64",
    "cb_person_cred_hist_length": "int64",
    "credit_score": "int64",
    "loan_status": "int64"
}

for columna, tipo in tipos_esperados.items():
    tipo_actual = str(df[columna].dtype)
    if tipo_actual != tipo:
        logging.warning(f"Columna {columna}: tipo esperado {tipo}, tipo actual {tipo_actual}")
        print(f"{columna}: esperado {tipo}, actual {tipo_actual}")
    else:
        print(f"{columna}: tipo correcto ({tipo})")

Todas las columnas esperadas están presentes
person_age: tipo correcto (int64)
person_income: tipo correcto (int64)
person_emp_exp: tipo correcto (int64)
loan_amnt: tipo correcto (int64)
cb_person_cred_hist_length: tipo correcto (int64)
credit_score: tipo correcto (int64)
loan_status: tipo correcto (int64)


In [6]:
# Columna para marcar registros inválidos y la razón
df["valido"] = True
df["motivo_rechazo"] = ""

# Validación 1: edad entre 18 y 100 años
mascara_edad = ~df["person_age"].between(18, 100)
df.loc[mascara_edad, "valido"] = False
df.loc[mascara_edad, "motivo_rechazo"] += "edad fuera de rango | "
logging.info(f"Registros con edad inválida: {mascara_edad.sum()}")
print(f"Registros con edad inválida: {mascara_edad.sum()}")

# Validación 2: credit_score entre 300 y 850
mascara_score = ~df["credit_score"].between(300, 850)
df.loc[mascara_score, "valido"] = False
df.loc[mascara_score, "motivo_rechazo"] += "credit_score fuera de rango | "
logging.info(f"Registros con credit_score inválido: {mascara_score.sum()}")
print(f"Registros con credit_score inválido: {mascara_score.sum()}")

# Validación 3: ingreso mayor a 0
mascara_income = df["person_income"] <= 0
df.loc[mascara_income, "valido"] = False
df.loc[mascara_income, "motivo_rechazo"] += "ingreso inválido | "
logging.info(f"Registros con ingreso inválido: {mascara_income.sum()}")
print(f"Registros con ingreso inválido: {mascara_income.sum()}")

# Validación 4: loan_percent_income entre 0 y 1
mascara_percent = ~df["loan_percent_income"].between(0, 1)
df.loc[mascara_percent, "valido"] = False
df.loc[mascara_percent, "motivo_rechazo"] += "loan_percent_income fuera de rango | "
logging.info(f"Registros con loan_percent_income inválido: {mascara_percent.sum()}")
print(f"Registros con loan_percent_income fuera de rango: {mascara_percent.sum()}")

print()
print(f"Total registros válidos: {df['valido'].sum()}")
print(f"Total registros inválidos: {(~df['valido']).sum()}")

Registros con edad inválida: 7
Registros con credit_score inválido: 0
Registros con ingreso inválido: 0
Registros con loan_percent_income fuera de rango: 0

Total registros válidos: 44993
Total registros inválidos: 7


In [7]:
# Separar registros válidos e inválidos
df_validos = df[df["valido"] == True].drop(columns=["valido", "motivo_rechazo"])
df_invalidos = df[df["valido"] == False]

# Registrar en el log
logging.info(f"Registros válidos: {len(df_validos)}")
logging.info(f"Registros inválidos: {len(df_invalidos)}")
print(f"Registros válidos: {len(df_validos)}")
print(f"Registros inválidos: {len(df_invalidos)}")

# Exportar ambos datasets para el Notebook 3
df_validos.to_csv("../data/processed/loans_validos.csv", index=False)
df_invalidos.to_csv("../data/processed/loans_invalidos.csv", index=False)

logging.info("Archivos exportados correctamente")
logging.info("Fin del proceso de validación")
print()
print("Archivos exportados a data/processed/")

Registros válidos: 44993
Registros inválidos: 7

Archivos exportados a data/processed/
